Boundary Flux Jacobian v3


Updates in v3
1. use exact formulation for slip wall boundary flux jacobian

Updates in v2

1. allow multiple eps inside the boundary flux jacobian function
- ghost_state_jacobian (dUg/dUi) --> use eps = e-7
- viscous_flux_jacobian --> use eps = e-8

Function 
- ghost_state
- riemann_invariant_bc --> to get ghost state for riemann invariant BC
- boundary_flux_jacobian (by finite difference)

Added function
- wall_flux_jacobian_3d --> to get exact slip wall flux jacobian

Boundary Conditions to implement
- slip wall/ symmetric BC
- noslip adiabatic wall
- riemann invariant (subsonic inflow and outflow, supersonic inflow and outflow

In [8]:
import numpy as np
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)   # required: JAX defaults to float32

import matplotlib.pyplot as plt

In [ ]:
# =============================================================================
# GLOBAL FLUX JACOBIAN ASSEMBLER — FINITE DIFFERENCE VERSION (v2)
# Self-contained: all dependencies included.
#
# Updates from v1:
#   (1) viscous_flux_jacobians_fd  — relative eps scaling  (default eps_visc  = 1e-8)
#   (2) ghost_state_jacobian_fd    — relative eps scaling  (default eps_ghost = 1e-6)
#   (3) boundary_flux_jacobian_fd  — exact non-frozen ghost chain rule:
#                                    J_bc = A⁺ + A⁻ @ dUg/dUi  (slip / riemann)
#   (4) assemble_global_jacobian_fd — both eps values exposed in signature
#
# To tune eps, change only the two arguments in assemble_global_jacobian_fd:
#     eps_visc  : FD step for viscous flux Jacobian    (default 1e-8)
#     eps_ghost : FD step for ghost state Jacobian     (default 1e-6)
# =============================================================================

import numpy as np
from scipy.sparse import lil_matrix


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — Inviscid flux Jacobian building blocks
# ─────────────────────────────────────────────────────────────────────────────

def compute_dFdU(U, normal, gamma=1.4):
    """Inviscid flux Jacobian dF/dU at state U in direction normal."""
    rho  = U[0]
    rho_u, rho_v, rho_w, rho_E = U[1], U[2], U[3], U[4]
    nx, ny, nz = normal[0], normal[1], normal[2]
    u = rho_u / rho;  v = rho_v / rho;  w = rho_w / rho
    E = rho_E / rho
    phi = 0.5*(gamma - 1.0)*(u**2 + v**2 + w**2)
    V   = nx*u + ny*v + nz*w
    a1  = gamma*E - phi
    a2  = gamma - 1.0
    a3  = gamma - 2.0
    A   = np.zeros((5, 5))
    A[0, 1] = nx;   A[0, 2] = ny;   A[0, 3] = nz
    A[1, 0] = nx*phi - u*V
    A[1, 1] = V - a3*nx*u;  A[1, 2] = ny*u - a2*nx*v;  A[1, 3] = nz*u - a2*nx*w;  A[1, 4] = a2*nx
    A[2, 0] = ny*phi - v*V
    A[2, 1] = nx*v - a2*ny*u;  A[2, 2] = V - a3*ny*v;  A[2, 3] = nz*v - a2*ny*w;  A[2, 4] = a2*ny
    A[3, 0] = nz*phi - w*V
    A[3, 1] = nx*w - a2*nz*u;  A[3, 2] = ny*w - a2*nz*v;  A[3, 3] = V - a3*nz*w;  A[3, 4] = a2*nz
    A[4, 0] = V*(phi - a1)
    A[4, 1] = a1*nx - a2*u*V;  A[4, 2] = a1*ny - a2*v*V;  A[4, 3] = a1*nz - a2*w*V;  A[4, 4] = gamma*V
    return A


def absolute_normal_jacobian(U, n, gamma=1.4):
    """Absolute-value normal Jacobian |A_n| (eqs. 3.6.16–3.6.26)."""
    U = np.asarray(U, dtype=float)
    n = np.asarray(n, dtype=float);  n = n / np.linalg.norm(n)
    rho = U[0];  vel = U[1:4] / rho;  E = U[4] / rho
    q2  = np.dot(vel, vel)
    p   = (gamma - 1.0)*rho*(E - 0.5*q2)
    c   = np.sqrt(gamma*p/rho)
    H   = E + p/rho
    qn  = np.dot(vel, n)
    M2  = q2/c**2;  Mn = qn/c;  g1 = gamma - 1.0

    # |qn| contribution
    mid = np.zeros((5, 5))
    mid[0, 0]   =  1.0 - 0.5*g1*M2
    mid[0, 1:4] =  (g1/c**2)*vel;   mid[0, 4] = -(g1/c**2)
    mid[1:4, 0]   = -0.5*g1*M2*vel + qn*n
    mid[1:4, 1:4] =  (g1/c**2)*np.outer(vel, vel) + np.eye(3) - np.outer(n, n)
    mid[1:4, 4]   = -(g1/c**2)*vel
    mid[4, 0]   =  qn**2 - 0.5*q2*(1.0 + 0.5*g1*M2)
    mid[4, 1:4] =  (1.0 + 0.5*g1*M2)*vel - qn*n;  mid[4, 4] = -0.5*g1*M2

    # |qn - c| contribution
    l1 = np.empty(5)
    l1[0] = 0.25*g1*M2 + 0.5*Mn;  l1[1:4] = -(g1/(2*c**2))*vel - n/(2*c);  l1[4] = g1/(2*c**2)
    r1 = np.empty(5)
    r1[0] = 1.0;  r1[1:4] = vel - c*n;  r1[4] = H - qn*c
    A1 = np.outer(r1, l1)

    # |qn + c| contribution
    l3 = np.empty(5)
    l3[0] = 0.25*g1*M2 - 0.5*Mn;  l3[1:4] = -(g1/(2*c**2))*vel + n/(2*c);  l3[4] = g1/(2*c**2)
    r3 = np.empty(5)
    r3[0] = 1.0;  r3[1:4] = vel + c*n;  r3[4] = H + qn*c
    A3 = np.outer(r3, l3)

    return abs(qn - c)*A1 + abs(qn)*mid + abs(qn + c)*A3


def roe_average(UL, UR, gamma=1.4):
    """Roe-averaged conservative state."""
    UL = np.asarray(UL, dtype=float);  UR = np.asarray(UR, dtype=float)
    rhoL, rhoR = UL[0], UR[0]
    velL = UL[1:4]/rhoL;  velR = UR[1:4]/rhoR
    EL = UL[4]/rhoL;      ER  = UR[4]/rhoR
    pL = (gamma-1.0)*rhoL*(EL - 0.5*np.dot(velL, velL))
    pR = (gamma-1.0)*rhoR*(ER - 0.5*np.dot(velR, velR))
    HL = EL + pL/rhoL;  HR = ER + pR/rhoR
    wL = np.sqrt(rhoL);  wR = np.sqrt(rhoR);  ws = wL + wR
    rho_roe = wL*wR
    vel_roe = (wL*velL + wR*velR)/ws
    H_roe   = (wL*HL   + wR*HR  )/ws
    q2_roe  = np.dot(vel_roe, vel_roe)
    E_roe   = (H_roe + (gamma-1.0)*0.5*q2_roe)/gamma
    U_roe   = np.empty(5)
    U_roe[0] = rho_roe;  U_roe[1:4] = rho_roe*vel_roe;  U_roe[4] = rho_roe*E_roe
    return U_roe


def inviscid_flux_jacobians(U_i, U_j, n, gamma=1.4):
    """
    Roe-split inviscid flux Jacobian blocks.
    Returns (J_L, J_R) = (A⁺, A⁻) at the Roe-averaged state.
    """
    U_roe     = roe_average(U_i, U_j, gamma)
    abs_A_roe = absolute_normal_jacobian(U_roe, n, gamma=gamma)
    J_L = 0.5*(compute_dFdU(U_i, n, gamma) + abs_A_roe)   # A⁺
    J_R = 0.5*(compute_dFdU(U_j, n, gamma) - abs_A_roe)   # A⁻
    return J_L, J_R


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — Viscous flux and Jacobian
# ─────────────────────────────────────────────────────────────────────────────

def cons_to_prim(U, gamma, R_gas):
    """Conservative → primitive variables."""
    rho = U[0]
    u, v, w = U[1]/rho, U[2]/rho, U[3]/rho
    E = U[4]/rho
    p = (gamma-1.0)*rho*(E - 0.5*(u*u + v*v + w*w))
    T = p/(rho*R_gas)
    return rho, u, v, w, p, T


def Fnv_numeric(U_i, U_j, n, ds,
                gamma=1.4, R_gas=287.0, Pr=0.72,
                mu0=1.716e-5, T0=273.15, Suth_C=110.4):
    """Numeric normal viscous flux F_n^v(U_i, U_j, n, ds)."""
    nx, ny, nz = n
    rho_L, u_L, v_L, w_L, p_L, T_L = cons_to_prim(U_i, gamma, R_gas)
    rho_R, u_R, v_R, w_R, p_R, T_R = cons_to_prim(U_j, gamma, R_gas)
    u_avg = (u_L+u_R)/2;  v_avg = (v_L+v_R)/2
    w_avg = (w_L+w_R)/2;  T_avg = (T_L+T_R)/2
    mu_avg  = mu0*(T0+Suth_C)/(T_avg+Suth_C)*(T_avg/T0)**1.5
    dudn = (u_R-u_L)/ds;  dvdn = (v_R-v_L)/ds
    dwdn = (w_R-w_L)/ds;  dTdn = (T_R-T_L)/ds
    div    = dudn*nx + dvdn*ny + dwdn*nz
    tau_xx = mu_avg*(2*dudn*nx - (2/3)*div)
    tau_yy = mu_avg*(2*dvdn*ny - (2/3)*div)
    tau_zz = mu_avg*(2*dwdn*nz - (2/3)*div)
    tau_xy = mu_avg*(dudn*ny + dvdn*nx)
    tau_xz = mu_avg*(dudn*nz + dwdn*nx)
    tau_yz = mu_avg*(dvdn*nz + dwdn*ny)
    tau_nx = tau_xx*nx + tau_xy*ny + tau_xz*nz
    tau_ny = tau_xy*nx + tau_yy*ny + tau_yz*nz
    tau_nz = tau_xz*nx + tau_yz*ny + tau_zz*nz
    tau_nn = tau_nx*u_avg + tau_ny*v_avg + tau_nz*w_avg
    kappa  = gamma*mu_avg/(Pr*(gamma-1.0))
    q_n    = -kappa*dTdn
    return np.array([0., -tau_nx, -tau_ny, -tau_nz, -tau_nn + q_n])


def viscous_flux_jacobians_fd(U_i, U_j, n, ds, eps,
                               gamma=1.4, R_gas=287.0, Pr=0.72,
                               mu0=1.716e-5, T0=273.15, Suth_C=110.4,
                               eps_abs=1e-14):
    """
    Forward FD viscous flux Jacobian with relative eps scaling.
    h_k = eps * max(|U[k]|, eps_abs)
    """
    U_i = np.asarray(U_i, dtype=float)
    U_j = np.asarray(U_j, dtype=float)
    F_base = Fnv_numeric(U_i, U_j, n, ds, gamma, R_gas, Pr, mu0, T0, Suth_C)
    J_i_fd = np.zeros((5, 5))
    J_j_fd = np.zeros((5, 5))
    for k in range(5):
        h_i = eps * max(abs(U_i[k]), eps_abs)
        h_j = eps * max(abs(U_j[k]), eps_abs)
        U_i_fwd = U_i.copy();  U_i_fwd[k] += h_i
        J_i_fd[:, k] = (Fnv_numeric(U_i_fwd, U_j, n, ds, gamma, R_gas, Pr, mu0, T0, Suth_C)
                         - F_base) / h_i
        U_j_fwd = U_j.copy();  U_j_fwd[k] += h_j
        J_j_fd[:, k] = (Fnv_numeric(U_i, U_j_fwd, n, ds, gamma, R_gas, Pr, mu0, T0, Suth_C)
                         - F_base) / h_j
    return J_i_fd, J_j_fd


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — Ghost state and its Jacobian
# ─────────────────────────────────────────────────────────────────────────────

def riemann_invariant_bc(U_int, n, gamma, u_b, v_b, w_b, T_b, P_b, R_gas=287.0):
    """Ghost cell state via Riemann invariant BC (4-regime selection)."""
    U_int = np.asarray(U_int, dtype=float)
    n     = np.asarray(n,     dtype=float);  n = n / np.linalg.norm(n)
    cv    = R_gas/(gamma - 1.0)
    rho_i = U_int[0];  vel_i = U_int[1:4]/rho_i;  E_i = U_int[4]/rho_i
    Vn_i  = np.dot(vel_i, n);  Vt_i = vel_i - Vn_i*n
    p_i   = (gamma-1.0)*rho_i*(E_i - 0.5*np.dot(vel_i, vel_i))
    T_i   = p_i/(rho_i*R_gas);  c_i = np.sqrt(gamma*R_gas*T_i)
    rho_b = P_b/(R_gas*T_b);  vel_b = np.array([u_b, v_b, w_b])
    Vn_b  = np.dot(vel_b, n);  Vt_b = vel_b - Vn_b*n
    c_b   = np.sqrt(gamma*R_gas*T_b)
    fac   = 2.0/(gamma - 1.0)
    Rp_int = Vn_i + fac*c_i;  Rm_int = Vn_i - fac*c_i
    Rp_b   = Vn_b + fac*c_b;  Rm_b   = Vn_b - fac*c_b
    Mn_i   = Vn_i/c_i
    if Mn_i <= -1.0:
        Vn_wall = Vn_b;  c_wall = c_b;  Vt_wall = Vt_b
        s_wall  = P_b/(rho_b**gamma)
    elif Mn_i >= 1.0:
        Vn_wall = Vn_i;  c_wall = c_i;  Vt_wall = Vt_i
        s_wall  = p_i/(rho_i**gamma)
    elif Vn_i < 0.0:
        Vn_wall = 0.5*(Rp_int + Rm_b);  c_wall = 0.25*(gamma-1.0)*(Rp_int - Rm_b)
        Vt_wall = Vt_b;  s_wall = P_b/(rho_b**gamma)
    else:
        Vn_wall = 0.5*(Rp_int + Rm_b);  c_wall = 0.25*(gamma-1.0)*(Rp_int - Rm_b)
        Vt_wall = Vt_i;  s_wall = p_i/(rho_i**gamma)
    if c_wall <= 0.0:
        raise ValueError(f"Non-physical c_wall={c_wall:.4f}")
    rho_wall = (c_wall**2/(gamma*s_wall))**(1.0/(gamma-1.0))
    p_wall   = s_wall*rho_wall**gamma
    T_wall   = p_wall/(rho_wall*R_gas)
    vel_wall = Vn_wall*n + Vt_wall
    E_wall   = cv*T_wall + 0.5*np.dot(vel_wall, vel_wall)
    U_wall_c = np.array([rho_wall, rho_wall*vel_wall[0],
                          rho_wall*vel_wall[1], rho_wall*vel_wall[2],
                          rho_wall*E_wall])
    return 2.0*U_wall_c - U_int


def ghost_state(U, bc_type, n, gamma=1.4, R_gas=287.0,
                u_b=0.0, v_b=0.0, w_b=0.0, T_b=300.0, P_b=101325.0):
    """Ghost cell state for slip / noslip / riemann BCs."""
    U = np.asarray(U, dtype=float)
    n = np.asarray(n, dtype=float);  n = n / np.linalg.norm(n)
    rho = U[0];  vel = U[1:4]/rho;  rho_E = U[4]
    if bc_type == 'slip':
        vn = np.dot(vel, n)
        return np.array([rho, *(rho*(vel - 2.0*vn*n)), rho_E])
    elif bc_type == 'noslip':
        return np.array([rho, *(-rho*vel), rho_E])
    elif bc_type == 'riemann':
        return riemann_invariant_bc(U, n, gamma, u_b, v_b, w_b, T_b, P_b, R_gas)
    else:
        raise ValueError(f"Unknown bc_type '{bc_type}'.")


def ghost_state_jacobian_fd(U_i, bc_type, n,
                             gamma=1.4, R_gas=287.0,
                             u_b=0.0, v_b=0.0, w_b=0.0,
                             T_b=288.15, P_b=101325.0,
                             eps=1e-6,
                             eps_abs=1e-14):
    """
    dU_ghost/dU_i (5×5) via forward FD with relative eps scaling.
    h_k = eps * max(|U_i[k]|, eps_abs)
    Differentiates the actual ghost_state() — consistent with assemble_residual.
    """
    U_i    = np.asarray(U_i, dtype=float)
    n      = np.asarray(n,   dtype=float);  n = n / np.linalg.norm(n)
    Ug_base = ghost_state(U_i, bc_type, n, gamma, R_gas, u_b, v_b, w_b, T_b, P_b)
    dUg     = np.zeros((5, 5))
    for k in range(5):
        h  = eps * max(abs(U_i[k]), eps_abs)
        ej = np.zeros(5);  ej[k] = h
        Ug_fwd    = ghost_state(U_i + ej, bc_type, n, gamma, R_gas, u_b, v_b, w_b, T_b, P_b)
        dUg[:, k] = (Ug_fwd - Ug_base) / h
    return dUg      # (5,5)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — Boundary flux Jacobian (non-frozen ghost)
# ─────────────────────────────────────────────────────────────────────────────

def boundary_flux_jacobian_fd(U_i, bc_type, n, ds,
                               eps_visc, eps_ghost,
                               gamma=1.4, R_gas=287.0,
                               Pr=0.72, mu0=1.716e-5,
                               T0=273.15, Suth_C=110.4,
                               u_b=0.0, v_b=0.0, w_b=0.0,
                               T_b=300.0, P_b=101325.0):
    """
    Boundary flux Jacobian dF_bc/dU_i  (5×5).

    slip / riemann — exact chain rule (non-frozen ghost):
        J_bc = A⁺ + A⁻ @ dUg/dUi
    noslip — viscous FD Jacobian.
    """
    U_i = np.asarray(U_i, dtype=float)
    n   = np.asarray(n,   dtype=float);  n = n / np.linalg.norm(n)

    if bc_type == 'riemann':
        U_g = ghost_state(U_i, 'riemann', n, gamma, R_gas, u_b, v_b, w_b, T_b, P_b)
    else:
        U_g = ghost_state(U_i, bc_type, n)

    if bc_type in ('slip', 'riemann'):
        J_L, J_R = inviscid_flux_jacobians(U_i, U_g, n, gamma)
        dUg_dUi  = ghost_state_jacobian_fd(
            U_i, bc_type, n,
            gamma=gamma, R_gas=R_gas,
            u_b=u_b, v_b=v_b, w_b=w_b,
            T_b=T_b, P_b=P_b,
            eps=eps_ghost)
        return J_L + J_R @ dUg_dUi

    elif bc_type == 'noslip':
        J_bc, _ = viscous_flux_jacobians_fd(
            U_i, U_g, n, max(ds, 1e-14),
            eps=eps_visc,
            gamma=gamma, R_gas=R_gas,
            Pr=Pr, mu0=mu0, T0=T0, Suth_C=Suth_C)
        return J_bc

    else:
        raise ValueError(f"Unknown bc_type '{bc_type}'.")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — Global assembler  ← main entry point
#
# Tune eps here:
#   eps_visc  = 1e-8   (viscous flux FD step)
#   eps_ghost = 1e-6   (ghost state FD step)
# ─────────────────────────────────────────────────────────────────────────────

def assemble_global_jacobian_fd(fortfile, U_list, coord, boundary_list,
                                 gamma=1.4, R_gas=287.0,
                                 viscous=False,
                                 Pr=0.72, mu0=1.716e-5,
                                 T0=273.15, Suth_C=110.4,
                                 eps_visc=1e-8,
                                 eps_ghost=1e-6):
    N       = U_list.shape[0]
    J       = lil_matrix((5*N, 5*N))
    visc_kw = dict(gamma=gamma, R_gas=R_gas, Pr=Pr, mu0=mu0, T0=T0, Suth_C=Suth_C)

    # ── Interior faces ────────────────────────────────────────────────────────
    for row in fortfile:
        i    = int(row[0]) - 1;  j = int(row[1]) - 1
        A_ij = np.asarray(row[2:5], dtype=float)
        Area = np.linalg.norm(A_ij)
        if Area < 1e-14:
            continue
        n_ij = A_ij / Area

        J_L, J_R = inviscid_flux_jacobians(U_list[i], U_list[j], n_ij, gamma)

        if viscous:
            ds      = max(abs(float(np.dot(coord[j] - coord[i], n_ij))), 1e-14)
            Jv_L, Jv_R = viscous_flux_jacobians_fd(
                U_list[i], U_list[j], n_ij, ds,
                eps=eps_visc, **visc_kw)
            J_L = J_L + Jv_L;  J_R = J_R + Jv_R

        ri = slice(5*i, 5*i+5);  ci = slice(5*i, 5*i+5)
        rj = slice(5*j, 5*j+5);  cj = slice(5*j, 5*j+5)
        
        J[ri, ci] += J_L * Area
        J[ri, cj] += J_R * Area
        J[rj, ci] -= J_L * Area
        J[rj, cj] -= J_R * Area

    # ── Boundary faces ────────────────────────────────────────────────────────
    bc_map = {0: 'riemann', 1: 'slip', 2: 'noslip'}

    for row in boundary_list:
        i       = int(row[0]) - 1
        A_bc = np.asarray(row[1:4], dtype=float)
        Area = np.linalg.norm(A_bc)
        ds   = float(row[4])
        if Area < 1e-14:
            continue
        n_bc = A_bc / Area
        bc_type = bc_map[int(row[5])]
        u_b, v_b, w_b = float(row[6]), float(row[7]), float(row[8])
        T_b, P_b      = float(row[9]),  float(row[10])

        J_bc = boundary_flux_jacobian_fd(
            U_list[i], bc_type, n_bc, ds,
            eps_visc=eps_visc,
            eps_ghost=eps_ghost,
            u_b=u_b, v_b=v_b, w_b=w_b,
            T_b=T_b, P_b=P_b,
            **visc_kw)

        ri = slice(5*i, 5*i+5)
        J[ri, ri] += J_bc * Area

    return J.tocsr()    # (5N × 5N) sparse CSR

In [2]:
def dV_dU(rho, u, v, w, gamma):
    """
    Jacobian of primitive V=[ρ,u,v,w,p] w.r.t. conservative U=[ρ,ρu,ρv,ρw,ρE]
    5×5 matrix (3-D extension of Eq. 4.32)
    """
    g1  = gamma - 1.0
    q2  = u**2 + v**2 + w**2
    return np.array([
        [ 1.0,       0.0,    0.0,    0.0,   0.0 ],   # ∂ρ/∂U
        [-u/rho,  1.0/rho,   0.0,    0.0,   0.0 ],   # ∂u/∂U
        [-v/rho,    0.0,  1.0/rho,   0.0,   0.0 ],   # ∂v/∂U
        [-w/rho,    0.0,    0.0,  1.0/rho,  0.0 ],   # ∂w/∂U
        [ 0.5*q2*g1, -u*g1, -v*g1, -w*g1,  g1  ],   # ∂p/∂U
    ])

def slip_wall_flux_jacobian(U_i, n, gamma=1.4):
    """
    Exact slip-wall boundary flux Jacobian  dF_bc/dU_i  (5×5).

    Slip-wall flux: F = [0, P*nx, P*ny, P*nz, 0]^T  (u_n = 0 → no convection)
    So  dF/dV|_W  is nonzero only in the pressure column (index 4):
        row 1 → nx,  row 2 → ny,  row 3 → nz,  all others 0

    Chain rule:  dF/dU = dF/dV|_W  @  dV/dU

    Parameters
    ----------
    U_i : (5,)  conservative state [ρ, ρu, ρv, ρw, ρE]
    n   : (3,)  outward unit normal  [nx, ny, nz]
    """
    U_i = np.asarray(U_i, dtype=float)
    n   = np.asarray(n,   dtype=float)
    n   = n / np.linalg.norm(n)           # ensure unit normal
    nx, ny, nz = n

    rho = U_i[0]
    u, v, w = U_i[1]/rho, U_i[2]/rho, U_i[3]/rho

    # dF/dV|_W — pressure-column-only, 5×5
    dFdV_W          = np.zeros((5, 5))
    dFdV_W[1, 4]    = nx
    dFdV_W[2, 4]    = ny
    dFdV_W[3, 4]    = nz
    # rows 0 and 4 stay zero (mass and energy fluxes vanish at slip wall)

    return dFdV_W @ dV_dU(rho, u, v, w, gamma)